# CIFAR-10 이미지 분류 CNN 모델 — TensorFlow Tensor 기반 코드

이 노트북은 CIFAR-10 이미지 데이터를 TensorFlow의 Tensor 연산 흐름으로 처리하고, CNN 모델을 학습·평가하는 예제입니다.

구성은 다음 순서로 진행됩니다.

1. 실행 환경 확인
2. 라이브러리 불러오기
3. 하이퍼파라미터 설정
4. CIFAR-10 데이터셋 준비
5. Tensor 데이터 전처리
6. 샘플 이미지 시각화
7. CNN 모델 설계
8. 손실함수와 최적화 알고리즘 설정
9. 모델 학습
10. 학습 과정 시각화
11. 최종 평가와 혼동행렬
12. 예측 결과 확인
13. 모델 저장과 불러오기

이 노트북에서 핵심 데이터 단위는 Tensor이다. Tensor는 숫자 데이터를 다차원 배열 형태로 표현하는 구조이며, 이미지 한 장은 `[높이, 너비, 채널]` 형태의 3차원 Tensor로 표현된다. 여러 이미지를 한 번에 처리할 때는 `[배치크기, 높이, 너비, 채널]` 형태의 4차원 Tensor가 사용된다.

## 1. 라이브러리 불러오기와 실행 환경 확인

TensorFlow는 Tensor 계산, 신경망 모델 구성, 자동 미분, 최적화, 학습 실행을 하나의 흐름으로 제공한다.  
NumPy는 배열 확인과 일부 후처리에 사용하고, Matplotlib은 이미지와 학습 그래프를 시각화하는 데 사용한다.

In [ ]:
# 운영체제 관련 설정을 위해 os 모듈을 불러옵니다.
import os

# 난수 고정과 일부 수치 처리를 위해 random 모듈을 불러옵니다.
import random

# 배열 연산과 혼동행렬 계산을 위해 NumPy 라이브러리를 불러옵니다.
import numpy as np

# 이미지와 그래프를 화면에 출력하기 위해 Matplotlib의 pyplot 모듈을 불러옵니다.
import matplotlib.pyplot as plt

# Tensor 연산, 신경망 모델 구성, 학습, 평가를 위해 TensorFlow 라이브러리를 불러옵니다.
import tensorflow as tf

# TensorFlow 안에 포함된 Keras API를 사용하기 위해 keras 별칭을 지정합니다.
keras = tf.keras

# Keras에서 신경망 계층을 만들 때 사용할 layers 모듈을 별칭으로 지정합니다.
layers = tf.keras.layers

# TensorFlow가 출력하는 정보성 로그를 줄여 노트북 화면을 깔끔하게 유지합니다.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# 현재 실행 중인 TensorFlow 버전을 출력합니다.
print("TensorFlow version:", tf.__version__)

# GPU 장치 목록을 확인합니다.
gpus = tf.config.list_physical_devices("GPU")

# GPU가 감지되었는지 여부를 출력합니다.
print("GPU devices:", gpus)

# GPU가 존재하는 경우에만 반복합니다.
for gpu in gpus:
    # GPU 메모리를 필요한 만큼만 점진적으로 할당하도록 설정합니다.
    tf.config.experimental.set_memory_growth(gpu, True)

## 2. 하이퍼파라미터 설정

하이퍼파라미터는 모델 학습 전에 직접 정하는 값이다.  
에포크 수는 전체 학습 데이터를 몇 번 반복할지 결정하고, 배치 크기는 한 번에 몇 장의 이미지를 처리할지 결정한다.  
학습률은 손실을 줄이기 위해 가중치를 한 번 수정할 때 이동하는 크기를 의미한다.

In [ ]:
# 실험 결과가 가능한 한 일정하게 나오도록 난수 시드를 하나의 값으로 고정합니다.
SEED = 111

# Python random 모듈의 난수 시드를 고정합니다.
random.seed(SEED)

# NumPy 난수 시드를 고정합니다.
np.random.seed(SEED)

# TensorFlow 난수 시드를 고정합니다.
tf.random.set_seed(SEED)

# 전체 학습 데이터를 반복해서 학습할 횟수를 지정합니다.
EPOCHS = 5

# 한 번의 학습 단계에서 사용할 이미지 개수를 지정합니다.
BATCH_SIZE = 64

# 최적화 알고리즘이 가중치를 업데이트할 때 사용할 학습률을 지정합니다.
LEARNING_RATE = 0.001

# CIFAR-10의 클래스 개수를 지정합니다.
NUM_CLASSES = 10

# 이미지 높이를 지정합니다.
IMG_HEIGHT = 32

# 이미지 너비를 지정합니다.
IMG_WIDTH = 32

# RGB 이미지의 채널 개수를 지정합니다.
IMG_CHANNELS = 3

# 입력 이미지 Tensor의 모양을 하나의 튜플로 저장합니다.
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# 설정한 하이퍼파라미터를 확인하기 위해 출력합니다.
print("EPOCHS:", EPOCHS)

# 설정한 배치 크기를 확인하기 위해 출력합니다.
print("BATCH_SIZE:", BATCH_SIZE)

# 설정한 학습률을 확인하기 위해 출력합니다.
print("LEARNING_RATE:", LEARNING_RATE)

# 입력 Tensor의 모양을 확인하기 위해 출력합니다.
print("INPUT_SHAPE:", INPUT_SHAPE)

## 3. CIFAR-10 데이터셋 준비

CIFAR-10은 32×32 크기의 컬러 이미지 데이터셋이다.  
이미지는 10개의 클래스 중 하나에 속한다.

클래스는 `airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`이다.

TensorFlow에서 불러온 이미지는 처음에 정수 픽셀값을 가지며, 학습에 사용하기 위해 실수 Tensor로 변환하고 `[0, 1]` 범위로 스케일링한다.

In [ ]:
# CIFAR-10 데이터셋을 TensorFlow Keras 데이터셋 API로 불러옵니다.
# x_train은 학습 이미지, y_train은 학습 이미지의 정답 라벨입니다.
# x_test는 평가 이미지, y_test는 평가 이미지의 정답 라벨입니다.
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# CIFAR-10 클래스 번호를 사람이 읽을 수 있는 이름으로 변환하기 위한 리스트를 정의합니다.
classes = [
    "airplane",    # 0번 클래스 이름입니다.
    "automobile",  # 1번 클래스 이름입니다.
    "bird",        # 2번 클래스 이름입니다.
    "cat",         # 3번 클래스 이름입니다.
    "deer",        # 4번 클래스 이름입니다.
    "dog",         # 5번 클래스 이름입니다.
    "frog",        # 6번 클래스 이름입니다.
    "horse",       # 7번 클래스 이름입니다.
    "ship",        # 8번 클래스 이름입니다.
    "truck"        # 9번 클래스 이름입니다.
]

# 학습 이미지 배열의 모양을 출력합니다.
print("학습 이미지 원본 모양:", x_train.shape)

# 학습 라벨 배열의 모양을 출력합니다.
print("학습 라벨 원본 모양:", y_train.shape)

# 평가 이미지 배열의 모양을 출력합니다.
print("평가 이미지 원본 모양:", x_test.shape)

# 평가 라벨 배열의 모양을 출력합니다.
print("평가 라벨 원본 모양:", y_test.shape)

# 첫 번째 학습 이미지의 데이터 타입을 출력합니다.
print("원본 이미지 데이터 타입:", x_train.dtype)

# 첫 번째 학습 라벨 값을 출력합니다.
print("첫 번째 학습 라벨 번호:", y_train[0][0])

# 첫 번째 학습 라벨 이름을 출력합니다.
print("첫 번째 학습 라벨 이름:", classes[y_train[0][0]])

## 4. Tensor 데이터 전처리

이미지 모델은 일반적으로 실수형 Tensor를 입력으로 사용한다.  
원본 픽셀값은 0부터 255까지의 정수이므로, `255.0`으로 나누어 0부터 1 사이의 실수값으로 변환한다.

라벨은 `(샘플 수, 1)` 형태로 불러와지므로, 학습과 평가에서 다루기 쉽도록 `(샘플 수,)` 형태의 1차원 Tensor로 바꾼다.

In [ ]:
# 학습 이미지 배열을 TensorFlow Tensor로 변환하고 데이터 타입을 float32로 바꿉니다.
x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)

# 평가 이미지 배열을 TensorFlow Tensor로 변환하고 데이터 타입을 float32로 바꿉니다.
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)

# 학습 이미지 픽셀값을 255.0으로 나누어 [0, 1] 범위로 변환합니다.
x_train = x_train / 255.0

# 평가 이미지 픽셀값을 255.0으로 나누어 [0, 1] 범위로 변환합니다.
x_test = x_test / 255.0

# 학습 라벨 배열을 int32 Tensor로 변환합니다.
y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)

# 평가 라벨 배열을 int32 Tensor로 변환합니다.
y_test = tf.convert_to_tensor(y_test, dtype=tf.int32)

# 학습 라벨 Tensor에서 크기가 1인 마지막 차원을 제거하여 1차원 라벨로 만듭니다.
y_train = tf.squeeze(y_train, axis=1)

# 평가 라벨 Tensor에서 크기가 1인 마지막 차원을 제거하여 1차원 라벨로 만듭니다.
y_test = tf.squeeze(y_test, axis=1)

# 전처리 후 학습 이미지 Tensor의 모양을 출력합니다.
print("전처리 후 학습 이미지 Tensor 모양:", x_train.shape)

# 전처리 후 학습 라벨 Tensor의 모양을 출력합니다.
print("전처리 후 학습 라벨 Tensor 모양:", y_train.shape)

# 전처리 후 평가 이미지 Tensor의 모양을 출력합니다.
print("전처리 후 평가 이미지 Tensor 모양:", x_test.shape)

# 전처리 후 평가 라벨 Tensor의 모양을 출력합니다.
print("전처리 후 평가 라벨 Tensor 모양:", y_test.shape)

# 전처리 후 이미지 Tensor의 데이터 타입을 출력합니다.
print("전처리 후 이미지 데이터 타입:", x_train.dtype)

# 전처리 후 라벨 Tensor의 데이터 타입을 출력합니다.
print("전처리 후 라벨 데이터 타입:", y_train.dtype)

## 5. `tf.data.Dataset` 생성

`tf.data.Dataset`은 Tensor 데이터를 학습에 적합한 형태로 공급하는 도구이다.  
학습 데이터는 섞기, 데이터 증강, 배치 묶기, 미리 준비하기 과정을 거친다.  
평가 데이터는 순서를 유지한 상태로 배치 단위로 묶어 사용한다.

In [ ]:
# 학습 이미지 Tensor와 학습 라벨 Tensor를 하나의 Dataset으로 묶습니다.
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))

# 평가 이미지 Tensor와 평가 라벨 Tensor를 하나의 Dataset으로 묶습니다.
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# 학습 데이터의 순서를 무작위로 섞어 모델이 데이터 순서에 의존하지 않도록 합니다.
train_dataset = train_dataset.shuffle(buffer_size=10000, seed=SEED, reshuffle_each_iteration=True)

# 학습 데이터를 배치 크기 단위로 묶습니다.
train_dataset = train_dataset.batch(BATCH_SIZE)

# 평가 데이터를 배치 크기 단위로 묶습니다.
test_dataset = test_dataset.batch(BATCH_SIZE)

# 학습 중 다음 배치를 미리 준비하여 데이터 공급 속도를 높입니다.
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# 평가 중 다음 배치를 미리 준비하여 데이터 공급 속도를 높입니다.
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

# 학습 Dataset의 원소 사양을 출력하여 이미지와 라벨 Tensor 구조를 확인합니다.
print("학습 Dataset 원소 구조:", train_dataset.element_spec)

# 평가 Dataset의 원소 사양을 출력하여 이미지와 라벨 Tensor 구조를 확인합니다.
print("평가 Dataset 원소 구조:", test_dataset.element_spec)

## 6. 샘플 이미지 확인

전처리된 이미지 Tensor는 `[0, 1]` 범위의 실수값을 가진다.  
Matplotlib은 이 범위의 RGB 이미지를 바로 출력할 수 있으므로 Tensor를 NumPy 배열로 변환한 뒤 화면에 표시한다.

In [ ]:
# 이미지 Tensor 여러 장을 격자 형태로 출력하는 함수를 정의합니다.
def show_images(images, labels, class_names, num_show=16):
    # 화면에 표시할 이미지 개수만큼 그림 크기를 지정합니다.
    plt.figure(figsize=(8, 8))

    # 지정한 이미지 개수만큼 반복합니다.
    for idx in range(num_show):
        # 4행 4열 격자에서 idx + 1번째 위치에 subplot을 만듭니다.
        plt.subplot(4, 4, idx + 1)

        # Tensor 이미지를 NumPy 배열로 변환합니다.
        img = images[idx].numpy()

        # 이미지 픽셀값을 [0, 1] 범위로 안전하게 제한합니다.
        img = np.clip(img, 0.0, 1.0)

        # 이미지를 화면에 출력합니다.
        plt.imshow(img)

        # 라벨 Tensor를 정수로 변환합니다.
        label_index = int(labels[idx].numpy())

        # 클래스 이름을 subplot 제목으로 표시합니다.
        plt.title(class_names[label_index])

        # 이미지 축 눈금을 숨깁니다.
        plt.axis("off")

    # subplot 간격을 자동으로 조정합니다.
    plt.tight_layout()

    # 완성된 그림을 화면에 출력합니다.
    plt.show()

# 학습 Dataset에서 첫 번째 배치를 가져옵니다.
sample_images, sample_labels = next(iter(train_dataset))

# 첫 번째 배치의 이미지 Tensor 모양을 출력합니다.
print("샘플 이미지 배치 모양:", sample_images.shape)

# 첫 번째 배치의 라벨 Tensor 모양을 출력합니다.
print("샘플 라벨 배치 모양:", sample_labels.shape)

# 샘플 이미지를 화면에 출력합니다.
show_images(sample_images, sample_labels, classes, num_show=16)

## 7. CNN 모델 설계

CNN은 이미지의 지역적인 패턴을 단계적으로 추출하는 모델 구조이다.  
합성곱 계층은 선, 모서리, 색상 변화 같은 특징을 찾고, 뒤쪽 계층으로 갈수록 더 복합적인 패턴을 학습한다.

이 모델은 다음 흐름으로 구성된다.

1. 입력 Tensor를 받는다.
2. 데이터 증강 계층으로 이미지 변형을 적용한다.
3. 합성곱 계층으로 특징맵을 만든다.
4. 배치 정규화로 출력 분포를 안정화한다.
5. ReLU 활성화 함수로 비선형성을 추가한다.
6. MaxPooling으로 특징맵 크기를 줄인다.
7. Flatten으로 1차원 벡터로 펼친다.
8. Dense 계층으로 분류 점수를 계산한다.
9. 마지막 Dense 계층에서 10개 클래스에 대한 logits를 출력한다.

출력층에는 Softmax를 직접 넣지 않는다. 손실함수에서 logits를 기준으로 계산하도록 설정한다.

In [ ]:
# CIFAR-10 분류를 위한 CNN 모델을 만드는 함수를 정의합니다.
def build_cifar10_cnn(input_shape=(32, 32, 3), num_classes=10):
    # 모델의 입력 Tensor 형태를 정의합니다.
    inputs = keras.Input(shape=input_shape, name="input_image")

    # 학습 데이터 다양성을 높이기 위해 좌우 반전 계층을 적용합니다.
    x = layers.RandomFlip("horizontal", name="random_flip")(inputs)

    # 학습 데이터 다양성을 높이기 위해 이미지 위치를 조금 이동하는 계층을 적용합니다.
    x = layers.RandomTranslation(height_factor=0.1, width_factor=0.1, name="random_translation")(x)

    # 첫 번째 합성곱 계층으로 32개의 특징맵을 생성합니다.
    x = layers.Conv2D(filters=32, kernel_size=3, padding="same", use_bias=False, name="conv2d_1")(x)

    # 첫 번째 합성곱 출력의 분포를 안정화합니다.
    x = layers.BatchNormalization(name="batch_norm_1")(x)

    # 첫 번째 합성곱 출력에 ReLU 활성화 함수를 적용합니다.
    x = layers.Activation("relu", name="relu_1")(x)

    # 32x32 특징맵을 16x16으로 줄입니다.
    x = layers.MaxPooling2D(pool_size=2, name="max_pool_1")(x)

    # 두 번째 합성곱 계층으로 64개의 특징맵을 생성합니다.
    x = layers.Conv2D(filters=64, kernel_size=3, padding="same", use_bias=False, name="conv2d_2")(x)

    # 두 번째 합성곱 출력의 분포를 안정화합니다.
    x = layers.BatchNormalization(name="batch_norm_2")(x)

    # 두 번째 합성곱 출력에 ReLU 활성화 함수를 적용합니다.
    x = layers.Activation("relu", name="relu_2")(x)

    # 16x16 특징맵을 8x8로 줄입니다.
    x = layers.MaxPooling2D(pool_size=2, name="max_pool_2")(x)

    # 세 번째 합성곱 계층으로 128개의 특징맵을 생성합니다.
    x = layers.Conv2D(filters=128, kernel_size=3, padding="same", use_bias=False, name="conv2d_3")(x)

    # 세 번째 합성곱 출력의 분포를 안정화합니다.
    x = layers.BatchNormalization(name="batch_norm_3")(x)

    # 세 번째 합성곱 출력에 ReLU 활성화 함수를 적용합니다.
    x = layers.Activation("relu", name="relu_3")(x)

    # 8x8 특징맵을 4x4로 줄입니다.
    x = layers.MaxPooling2D(pool_size=2, name="max_pool_3")(x)

    # 다차원 특징맵 Tensor를 1차원 벡터로 펼칩니다.
    x = layers.Flatten(name="flatten")(x)

    # 펼쳐진 특징 벡터를 256개 뉴런의 Dense 계층에 전달합니다.
    x = layers.Dense(units=256, use_bias=False, name="dense_1")(x)

    # Dense 계층 출력의 분포를 안정화합니다.
    x = layers.BatchNormalization(name="batch_norm_4")(x)

    # Dense 계층 출력에 ReLU 활성화 함수를 적용합니다.
    x = layers.Activation("relu", name="relu_4")(x)

    # 과적합을 줄이기 위해 학습 중 일부 뉴런 출력을 무작위로 끕니다.
    x = layers.Dropout(rate=0.3, name="dropout")(x)

    # 10개 클래스에 대한 최종 점수인 logits를 출력합니다.
    outputs = layers.Dense(units=num_classes, name="logits")(x)

    # 입력 Tensor와 출력 Tensor를 연결하여 Keras 모델 객체를 생성합니다.
    model = keras.Model(inputs=inputs, outputs=outputs, name="cifar10_tensor_cnn")

    # 완성된 모델 객체를 반환합니다.
    return model

# CNN 모델 객체를 생성합니다.
model = build_cifar10_cnn(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES)

# 모델 구조와 파라미터 개수를 요약하여 출력합니다.
model.summary()

## 8. 손실함수와 최적화 알고리즘 설정

다중 클래스 분류에서는 실제 정답 클래스 번호와 모델의 클래스별 점수를 비교해야 한다.  
정답 라벨이 원-핫 벡터가 아니라 정수 번호 형태이므로 `SparseCategoricalCrossentropy`를 사용한다.

모델의 마지막 출력은 확률이 아니라 logits이다.  
따라서 손실함수에서 `from_logits=True`로 설정하여 내부적으로 안정적인 확률 변환과 손실 계산이 함께 이루어지게 한다.

최적화 알고리즘은 손실을 줄이는 방향으로 모델의 가중치를 반복적으로 업데이트한다.

In [ ]:
# 정수 라벨과 logits 출력을 비교하는 손실함수를 생성합니다.
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# AdamW 최적화 알고리즘을 생성합니다.
optimizer = keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE, # 가중치 업데이트 크기를 결정하는 학습률을 지정합니다.
    weight_decay=1e-4            # 가중치가 지나치게 커지는 것을 완화하기 위한 감쇠 값을 지정합니다.
)

# 모델 학습 과정에서 확인할 평가 지표를 리스트로 정의합니다.
metrics = [
    keras.metrics.SparseCategoricalAccuracy(name="accuracy") # 정수 라벨 기준 정확도를 계산합니다.
]

# 모델에 최적화 알고리즘, 손실함수, 평가 지표를 연결합니다.
model.compile(
    optimizer=optimizer, # 모델 가중치를 업데이트할 최적화 알고리즘을 지정합니다.
    loss=loss_fn,        # 예측값과 정답값의 차이를 계산할 손실함수를 지정합니다.
    metrics=metrics      # 학습과 평가 중 출력할 지표를 지정합니다.
)

## 9. 모델 학습

모델 학습은 Tensor 배치를 반복적으로 입력하여 진행된다.  
각 배치마다 모델은 예측값을 만들고, 손실함수는 예측값과 정답값의 차이를 계산한다.  
자동 미분은 손실을 줄이기 위해 각 가중치를 어느 방향으로 수정해야 하는지 계산하고, 최적화 알고리즘은 그 결과를 이용해 가중치를 업데이트한다.

In [ ]:
# 학습률을 에포크 중간에 조정하기 위한 콜백을 생성합니다.
lr_callback = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", # 평가 손실을 기준으로 학습률 조정 여부를 판단합니다.
    factor=0.5,         # 개선이 부족할 때 학습률을 절반으로 줄입니다.
    patience=2,         # 2 에포크 동안 개선이 부족하면 학습률을 줄입니다.
    verbose=1           # 학습률이 변경될 때 메시지를 출력합니다.
)

# 학습 시간이 길어지는 상황을 줄이기 위한 조기 종료 콜백을 생성합니다.
early_stop_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",              # 평가 손실을 기준으로 조기 종료 여부를 판단합니다.
    patience=5,                      # 5 에포크 동안 개선이 부족하면 학습을 중단합니다.
    restore_best_weights=True        # 가장 좋은 평가 손실을 기록한 가중치로 되돌립니다.
)

# 모델 학습을 실행하고 에포크별 기록을 history 객체에 저장합니다.
history = model.fit(
    train_dataset,                         # 학습용 Tensor Dataset을 입력합니다.
    validation_data=test_dataset,          # 평가용 Tensor Dataset을 입력합니다.
    epochs=EPOCHS,                         # 전체 학습 반복 횟수를 지정합니다.
    callbacks=[lr_callback, early_stop_callback] # 학습률 조정과 조기 종료 콜백을 적용합니다.
)

## 10. 학습 과정 시각화

학습 손실과 평가 손실을 비교하면 모델이 학습 데이터를 잘 따라가고 있는지 확인할 수 있다.  
학습 정확도와 평가 정확도를 비교하면 새로운 데이터에 대한 예측 성능을 확인할 수 있다.

In [ ]:
# history 객체에서 학습 손실 기록을 가져옵니다.
train_loss = history.history["loss"]

# history 객체에서 평가 손실 기록을 가져옵니다.
val_loss = history.history["val_loss"]

# history 객체에서 학습 정확도 기록을 가져옵니다.
train_acc = history.history["accuracy"]

# history 객체에서 평가 정확도 기록을 가져옵니다.
val_acc = history.history["val_accuracy"]

# 실제로 수행된 에포크 수를 기준으로 x축 값을 생성합니다.
epochs_range = range(1, len(train_loss) + 1)

# 손실 그래프를 그릴 그림을 생성합니다.
plt.figure(figsize=(8, 5))

# 학습 손실 곡선을 그립니다.
plt.plot(epochs_range, train_loss, marker="o", label="Train Loss")

# 평가 손실 곡선을 그립니다.
plt.plot(epochs_range, val_loss, marker="o", label="Validation Loss")

# 그래프 제목을 지정합니다.
plt.title("Loss Curve")

# x축 이름을 지정합니다.
plt.xlabel("Epoch")

# y축 이름을 지정합니다.
plt.ylabel("Loss")

# 범례를 표시합니다.
plt.legend()

# 값 변화를 보기 쉽도록 격자를 표시합니다.
plt.grid(True)

# 완성된 그래프를 화면에 출력합니다.
plt.show()

# 정확도 그래프를 그릴 그림을 생성합니다.
plt.figure(figsize=(8, 5))

# 학습 정확도 곡선을 그립니다.
plt.plot(epochs_range, train_acc, marker="o", label="Train Accuracy")

# 평가 정확도 곡선을 그립니다.
plt.plot(epochs_range, val_acc, marker="o", label="Validation Accuracy")

# 그래프 제목을 지정합니다.
plt.title("Accuracy Curve")

# x축 이름을 지정합니다.
plt.xlabel("Epoch")

# y축 이름을 지정합니다.
plt.ylabel("Accuracy")

# 범례를 표시합니다.
plt.legend()

# 값 변화를 보기 쉽도록 격자를 표시합니다.
plt.grid(True)

# 완성된 그래프를 화면에 출력합니다.
plt.show()

## 11. 최종 평가와 혼동행렬

최종 평가는 학습이 끝난 모델이 평가 데이터에서 어느 정도 성능을 내는지 확인하는 단계이다.  
혼동행렬은 실제 클래스와 예측 클래스의 관계를 표로 보여준다.  
행은 실제 정답 클래스이고, 열은 모델이 예측한 클래스이다.

In [ ]:
# 평가 Dataset 전체에 대해 손실과 정확도를 계산합니다.
final_loss, final_accuracy = model.evaluate(test_dataset, verbose=0)

# 최종 평가 손실을 출력합니다.
print(f"최종 평가 손실: {final_loss:.4f}")

# 최종 평가 정확도를 백분율 형태로 출력합니다.
print(f"최종 평가 정확도: {final_accuracy * 100:.2f}%")

# 평가 Dataset 전체에 대해 클래스별 logits 예측값을 계산합니다.
logits = model.predict(test_dataset, verbose=0)

# logits에서 가장 큰 값을 가진 클래스 인덱스를 예측 클래스로 변환합니다.
pred_labels = tf.argmax(logits, axis=1, output_type=tf.int32)

# 평가 라벨 Tensor를 그대로 사용하기 위해 y_test를 정수 Tensor로 변환합니다.
true_labels = tf.cast(y_test, tf.int32)

# 실제 라벨과 예측 라벨을 이용해 10x10 혼동행렬 Tensor를 계산합니다.
confusion_tensor = tf.math.confusion_matrix(
    labels=true_labels,        # 실제 정답 라벨 Tensor를 입력합니다.
    predictions=pred_labels,   # 모델이 예측한 라벨 Tensor를 입력합니다.
    num_classes=NUM_CLASSES    # 클래스 개수를 지정합니다.
)

# 혼동행렬 Tensor를 NumPy 배열로 변환하여 출력과 시각화에 사용합니다.
confusion = confusion_tensor.numpy()

# 혼동행렬을 출력합니다.
print("\n혼동행렬:")
print(confusion)

# 클래스별 정확도를 출력하기 위한 제목을 표시합니다.
print("\n클래스별 정확도:")

# 클래스 개수만큼 반복합니다.
for i, class_name in enumerate(classes):
    # i번째 클래스의 전체 실제 샘플 개수를 계산합니다.
    class_total = confusion[i].sum()

    # i번째 클래스에서 정확히 예측한 샘플 개수를 계산합니다.
    class_correct = confusion[i, i]

    # 전체 개수가 0보다 크면 정확도를 계산하고, 그렇지 않으면 0으로 처리합니다.
    class_acc = class_correct / class_total if class_total > 0 else 0.0

    # 클래스 이름과 클래스별 정확도를 출력합니다.
    print(f"{class_name:10s}: {class_acc * 100:6.2f}%")

In [ ]:
# 혼동행렬을 시각화할 그림을 생성합니다.
plt.figure(figsize=(9, 8))

# 혼동행렬 값을 이미지 형태로 표시합니다.
plt.imshow(confusion, interpolation="nearest")

# 그래프 제목을 지정합니다.
plt.title("Confusion Matrix")

# 색상 막대를 추가하여 값의 크기를 색상으로 확인할 수 있게 합니다.
plt.colorbar()

# 클래스 개수만큼 눈금 위치를 생성합니다.
tick_marks = np.arange(len(classes))

# x축에 예측 클래스 이름을 표시합니다.
plt.xticks(tick_marks, classes, rotation=45)

# y축에 실제 클래스 이름을 표시합니다.
plt.yticks(tick_marks, classes)

# y축 이름을 지정합니다.
plt.ylabel("True Label")

# x축 이름을 지정합니다.
plt.xlabel("Predicted Label")

# 혼동행렬의 행 개수만큼 반복합니다.
for i in range(confusion.shape[0]):
    # 혼동행렬의 열 개수만큼 반복합니다.
    for j in range(confusion.shape[1]):
        # 각 칸의 값을 텍스트로 표시합니다.
        plt.text(j, i, str(confusion[i, j]), ha="center", va="center")

# 그래프 요소가 겹치지 않도록 레이아웃을 조정합니다.
plt.tight_layout()

# 완성된 혼동행렬 그래프를 화면에 출력합니다.
plt.show()

## 12. 예측 결과 확인

평가 이미지 일부를 모델에 입력하고, 예측 클래스와 실제 클래스를 함께 확인한다.  
모델 출력 logits는 `tf.argmax`를 이용해 가장 큰 점수를 가진 클래스 번호로 변환한다.

In [ ]:
# 평가 Dataset에서 첫 번째 배치를 가져옵니다.
test_images_batch, test_labels_batch = next(iter(test_dataset))

# 첫 번째 평가 배치에 대해 모델의 logits를 계산합니다.
test_logits = model(test_images_batch, training=False)

# logits에서 가장 큰 값을 가진 클래스 번호를 예측값으로 선택합니다.
test_pred_labels = tf.argmax(test_logits, axis=1, output_type=tf.int32)

# 화면에 출력할 이미지 개수를 지정합니다.
num_show = 8

# 예측 결과를 표시할 그림을 생성합니다.
plt.figure(figsize=(14, 4))

# 지정한 이미지 개수만큼 반복합니다.
for idx in range(num_show):
    # 한 줄에 여러 이미지를 배치하기 위한 subplot을 생성합니다.
    plt.subplot(1, num_show, idx + 1)

    # 이미지 Tensor를 NumPy 배열로 변환합니다.
    img = test_images_batch[idx].numpy()

    # 이미지 픽셀값을 [0, 1] 범위로 안전하게 제한합니다.
    img = np.clip(img, 0.0, 1.0)

    # 이미지를 화면에 출력합니다.
    plt.imshow(img)

    # 실제 라벨 번호를 정수로 변환합니다.
    true_idx = int(test_labels_batch[idx].numpy())

    # 예측 라벨 번호를 정수로 변환합니다.
    pred_idx = int(test_pred_labels[idx].numpy())

    # 실제 클래스와 예측 클래스를 제목으로 표시합니다.
    plt.title(f"P:{classes[pred_idx]}\nT:{classes[true_idx]}")

    # 이미지 축 눈금을 숨깁니다.
    plt.axis("off")

# subplot 사이의 간격을 자동으로 조정합니다.
plt.tight_layout()

# 완성된 예측 결과 그림을 화면에 출력합니다.
plt.show()

## 13. 모델 저장과 불러오기

학습이 끝난 모델은 파일로 저장할 수 있다.  
저장된 모델에는 모델 구조, 가중치, 컴파일 정보가 포함되므로 나중에 다시 불러와 평가나 예측에 사용할 수 있다.

In [ ]:
# 모델을 저장할 파일 경로를 지정합니다.
MODEL_PATH = "cifar10_tensor_cnn.keras"

# 학습된 모델 전체를 Keras 형식으로 저장합니다.
model.save(MODEL_PATH)

# 모델 저장 완료 메시지를 출력합니다.
print("모델 저장 완료:", MODEL_PATH)

# 저장된 Keras 모델을 다시 불러옵니다.
loaded_model = keras.models.load_model(MODEL_PATH)

# 불러온 모델의 구조를 요약하여 출력합니다.
loaded_model.summary()

# 불러온 모델을 평가 Dataset으로 다시 평가합니다.
loaded_loss, loaded_accuracy = loaded_model.evaluate(test_dataset, verbose=0)

# 불러온 모델의 평가 손실을 출력합니다.
print(f"불러온 모델 평가 손실: {loaded_loss:.4f}")

# 불러온 모델의 평가 정확도를 출력합니다.
print(f"불러온 모델 평가 정확도: {loaded_accuracy * 100:.2f}%")

## 14. 성능 개선 실험 방향

정확도를 더 높이려면 다음 항목을 조정해 볼 수 있다.

1. `EPOCHS`를 20 이상으로 늘린다.
2. `BATCH_SIZE`를 128로 변경한다.
3. 합성곱 계층 수를 늘린다.
4. `LEARNING_RATE`를 `0.0005`, `0.0003` 등으로 조정한다.
5. `RandomRotation`, `RandomZoom`, `RandomContrast` 같은 데이터 증강 계층을 추가한다.
6. Dropout 비율을 조정한다.
7. BatchNormalization 위치와 계층 구성을 실험한다.

모델이 복잡해질수록 학습 시간이 늘어나므로 GPU 런타임에서 실행하는 것이 좋다.